# M0.4 · Olasılık ve MLE özeti

ML'deki her kayıp fonksiyonunun arkasındaki fikir: maximum likelihood. Bir yazı-tura, bir Gauss ve bir zar için MLE'yi elle buluyor, saf Python'da log-likelihood yazıyor, NumPy'da bir ızgara üzerinde döngüsüz arıyor, `torch.distributions` ile gradyanla optimize ediyor ve en sonda bootstrap'ı tek tensör işlemiyle yapıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m0-4-probability.html](https://m-gonul.github.io/pytorch-lab/m0-4-probability.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m0_4_probability.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch/statistics kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"statistics\", \"pandas\"}))\n\n    def mv():\n        close([need(ns, \"mean\")([2, 4, 6]), need(ns, \"var\")([2, 4, 6]), need(ns, \"var\")([2, 4, 6], ddof=1)], [4, 8 / 3, 4])\n        rng = random.Random(0)\n        xs = [rng.gauss(3, 2) for _ in range(50)]\n        m = sum(xs) / 50\n        close(need(ns, \"var\")(xs), sum((v - m) ** 2 for v in xs) / 50, what=\"50 örnek MLE varyansı\")\n    R.test(\"mean, var (ddof = 0 ve 1)\", mv)\n\n    def ll():\n        f = need(ns, \"bernoulli_ll\")\n        close(f([1, 0, 1, 1], 0.75), 3 * math.log(0.75) + math.log(0.25))\n        ensure(f([1, 1], 0.0) == -math.inf, \"p = 0 iken tura gelmişse −inf bekleniyordu\")\n        ensure(f([0, 0], 1.0) == -math.inf, \"p = 1 iken yazı gelmişse −inf bekleniyordu\")\n        close(f([1, 1], 1.0), 0.0, what=\"p = 1, hep tura\")\n    R.test(\"bernoulli_ll ve p ∈ {0, 1} uç durumları\", ll)\n\n    def underflow():\n        rng = random.Random(1)\n        ys = [int(rng.random() < 0.6) for _ in range(2000)]\n        prod = math.prod(0.6 if y else 0.4 for y in ys)\n        v = need(ns, \"bernoulli_ll\")(ys, 0.6)\n        ensure(prod == 0.0 and math.isfinite(v), \"çarpım 0'a düşmeli, log toplamı sonlu kalmalı\")\n        return f\"2000 atış: çarpım = {prod}, log-likelihood = {v:.1f}\"\n    R.test(\"log toplamı alt taşmadan korur\", underflow)\n\n    def cat():\n        close(need(ns, \"categorical_mle\")([1, 3, 3, 6, 3, 1], 6), [2 / 6, 0, 3 / 6, 0, 0, 1 / 6])\n        p = need(ns, \"categorical_mle\")([2] * 5, 3)\n        close(p, [0, 1, 0], what=\"hep 2\")\n    R.test(\"categorical_mle\", cat)\n",
 "c": "import numpy as np\nimport pandas as pd\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok (comprehension dahil)\", lambda: no_loops(src, allow_comprehension=False))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n\n    def g():\n        X = rng.normal(loc=[1, -2, 5], scale=[1, 3, 0.5], size=(200, 3))\n        mu, var = need(ns, \"gaussian_mle\")(X)\n        close(mu, X.mean(0)); close(var, X.var(0))\n        close(need(ns, \"gaussian_mle\")(np.array([[2.0], [4.0], [6.0]]))[1], [8 / 3], what=\"elle örnek varyans\")\n    R.test(\"gaussian_mle: sütun başına ortalama ve MLE varyansı\", g)\n\n    def gs():\n        grid = np.linspace(0.01, 0.99, 99)\n        close(need(ns, \"grid_search_p\")(np.array([1, 0, 1, 1]), grid), 0.75, atol=1e-9)\n        ys = (rng.random(300) < 0.31).astype(int)\n        p = need(ns, \"grid_search_p\")(ys, grid)\n        ensure(abs(p - ys.mean()) <= 0.0051, f\"ızgara sonucu {p}, veri oranı {ys.mean():.3f}\")\n        return f\"300 atış: ızgara {p:.2f}, kapalı form {ys.mean():.3f}\"\n    R.test(\"grid_search_p kapalı forma (oran) en yakın ızgara noktası\", gs)\n\n    def cs():\n        df = pd.DataFrame({\"sinif\": [\"a\", \"a\", \"b\", \"b\", \"b\"], \"boy\": [1.0, 3.0, 2.0, 4.0, 6.0]})\n        out = need(ns, \"class_stats\")(df, \"sinif\", \"boy\")\n        ensure(isinstance(out, pd.DataFrame), \"DataFrame bekleniyordu\")\n        for c in (\"count\", \"mean\", \"var\"):\n            ensure(c in out.columns, f\"`{c}` sütunu yok\")\n        close(out.loc[[\"a\", \"b\"], [\"count\", \"mean\", \"var\"]].to_numpy(dtype=float), [[2, 2, 1], [3, 4, 8 / 3]])\n    R.test(\"class_stats: count, mean, MLE var\", cs)\n",
 "d": "import math\nimport torch\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"torch.distributions kullanılıyor\", lambda: ensure(uses_any(src, [\"Normal\", \"log_prob\"]), \"Normal(...).log_prob ile yaz\"))\n\n    def nll():\n        x = torch.tensor([2.0, 4.0, 6.0], dtype=torch.float64)\n        v = need(ns, \"normal_nll\")(x, torch.tensor(4.0, dtype=torch.float64), torch.tensor(1.0, dtype=torch.float64))\n        close(v, 0.5 * math.log(2 * math.pi) + (4 + 0 + 4) / 2 / 3)\n        v2 = need(ns, \"normal_nll\")(x, torch.tensor(3.0, dtype=torch.float64), torch.tensor(2.0, dtype=torch.float64))\n        close(v2, 0.5 * math.log(2 * math.pi) + math.log(2) + (1 + 1 + 9) / (2 * 4) / 3, what=\"μ = 3, σ = 2\")\n    R.test(\"normal_nll: elle formülle aynı (ortalama)\", nll)\n\n    def fit():\n        g = torch.Generator().manual_seed(0)\n        x = 3.0 + 1.7 * torch.randn(400, generator=g)\n        mu, sigma = need(ns, \"fit_normal\")(x)\n        close(mu, x.mean(), atol=2e-3, rtol=0, what=\"μ̂\")\n        close(sigma, x.var(unbiased=False).sqrt(), atol=2e-3, rtol=0, what=\"σ̂ (MLE)\")\n        ensure(float(sigma) > 0, \"σ pozitif olmalı\")\n        return f\"gradyanla μ̂ = {float(mu):.4f}, σ̂ = {float(sigma):.4f}; kapalı form {float(x.mean()):.4f}, {float(x.std(unbiased=False)):.4f}\"\n    R.test(\"fit_normal kapalı form MLE'ye ulaşıyor\", fit)\n",
 "e": "import torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(7)\n    x = torch.randn(64, generator=g, dtype=torch.float64) * 3 + 1\n\n    def bs():\n        m = need(ns, \"bootstrap_means\")(x, 20000, torch.Generator().manual_seed(1))\n        ensure(tuple(m.shape) == (20000,), f\"shape {tuple(m.shape)}\")\n        ensure(float(m.min()) >= float(x.min()) and float(m.max()) <= float(x.max()), \"ortalamalar verinin aralığı dışında\")\n        se = float(x.std(unbiased=False)) / 64 ** 0.5\n        ensure(abs(float(m.mean()) - float(x.mean())) < 4 * se / 20000 ** 0.5 + 1e-3, \"bootstrap ortalamalarının ortalaması x̄'dan uzak\")\n        ensure(abs(float(m.std()) / se - 1) < 0.05, f\"bootstrap std {float(m.std()):.4f}, teorik σ̂/√n {se:.4f}\")\n        return f\"bootstrap std {float(m.std()):.4f} ≈ σ̂/√n = {se:.4f}\"\n    R.test(\"bootstrap: standart hata ≈ σ̂/√n\", bs)\n\n    def det():\n        a = need(ns, \"bootstrap_means\")(x, 50, torch.Generator().manual_seed(3))\n        b = need(ns, \"bootstrap_means\")(x, 50, torch.Generator().manual_seed(3))\n        ensure(torch.equal(a, b), \"aynı seed farklı sonuç verdi: verilen generator kullanılmıyor\")\n    R.test(\"verilen generator kullanılıyor (tekrarlanabilir)\", det)\n\n    def bm():\n        X = torch.randn(30, 12, generator=g, dtype=torch.float64)\n        mu, var = need(ns, \"batched_gaussian_mle\")(X)\n        close(mu, X.mean(1)); close(var, X.var(1, unbiased=False))\n    R.test(\"batched_gaussian_mle: satır başına\", bm)\n    if \"bootstrap_means\" not in ns:\n        return\n    xb = torch.randn(500)\n\n    def loop():\n        n = xb.shape[0]\n        gg = torch.Generator().manual_seed(0)\n        return torch.stack([xb[torch.randint(n, (n,), generator=gg)].mean() for _ in range(2000)])\n    t_loop = bench(loop, repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"bootstrap_means\"](xb, 2000, torch.Generator().manual_seed(0)), repeat=3)\n    R.table(\"2000 bootstrap örneği (n = 500)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for b: randint + mean\", fmt_time(t_loop), \"1×\"],\n             [\"senin bootstrap_means\", fmt_time(t_vec), f\"{t_loop / t_vec:,.0f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m0_4_probability as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir model, verinin hangi süreçten geldiğine dair bir varsayımdır. **Likelihood** bu varsayımın gözlenen veriye ne kadar olasılık verdiğidir; parametreler $\theta$ için

$$
L(\theta)=\prod_{i=1}^n p(x_i\mid\theta),\qquad
\ell(\theta)=\log L(\theta)=\sum_{i=1}^n \log p(x_i\mid\theta).
$$

Maximum likelihood estimation veriyi en olası kılan $\theta$'yı seçer: $\hat\theta=\arg\max_\theta \ell(\theta)$. Optimizasyon kütüphaneleri minimize ettiği için pratikte negatif log-likelihood (NLL) minimize edilir; ML'deki kayıpların büyük kısmı budur:

| varsayım | NLL | adı |
|---|---|---|
| $y\sim\mathcal N(\hat y,\sigma^2)$ | $\frac{1}{2\sigma^2}(y-\hat y)^2+c$ | MSE (M1.1) |
| $y\sim\text{Bernoulli}(\sigma(z))$ | $-[y\log p+(1-y)\log(1-p)]$ | BCE (M1.4) |
| $y\sim\text{Kategorik}(\text{softmax}(z))$ | $-\log p_y$ | cross-entropy (M1.5) |

Üç dağılımın MLE'si kapalı formda: Bernoulli için $\hat p=\bar y$, Gauss için $\hat\mu=\bar x$ ve $\hat\sigma^2=\frac1n\sum(x_i-\bar x)^2$, kategorik için $\hat p_k=n_k/n$.

## 1 · Elle

<!-- çözüm -->

Önce olabilirliği iki farklı $p$ için doğrudan çarpımla hesapla: $p=0.75$ daha yüksek çıkmalı, çünkü veride 4 atışın 3'ü tura. Sonra $\log L$'nin türevini sıfıra eşitle:

$$
\ell(p)=k\ln p+(n-k)\ln(1-p),\qquad \ell'(p)=\frac kp-\frac{n-k}{1-p}=0\;\Rightarrow\;\hat p=\frac kn .
$$

Gauss'ta ortalama ve iki varyansı karşılaştır. Zarda 2 hiç gelmediği için MLE ona **sıfır** olasılık veriyor: model, hiç görmediği bir sonucu imkânsız sayıyor. Bu, küçük veride MLE'nin aşırı uyumunun en sade örneği.

Tablo: $\ln 0.75\approx-0.28768$, $\ln 0.25\approx-1.38629$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Yazı-tura: $y=[1,0,1,1]$ (1 = tura), $y_i\sim\text{Bernoulli}(p)$**

- `lik05`: $L(p=0.5)=\prod_i p^{y_i}(1-p)^{1-y_i}$
- `lik075`: $L(p=0.75)$
- `phat`: $\hat p_{\text{MLE}}$
- `ll`: $\log L(\hat p)=3\ln0.75+\ln0.25$

**Gauss: $x=[2,4,6]$, $x_i\sim\mathcal N(\mu,\sigma^2)$**

- `mu`: $\hat\mu$
- `var_mle`: $\hat\sigma^2_{\text{MLE}}=\frac1n\sum(x_i-\hat\mu)^2$
- `var_unb`: Yansız $s^2=\frac1{n-1}\sum(x_i-\hat\mu)^2$

**Zar: atışlar $[1,3,3,6,3,1]$; $\hat p_k=\frac{\#k}{n}$**

- `p1`: $\hat p_1$
- `p3`: $\hat p_3$
- `p2`: $\hat p_2$

In [ ]:
cevaplar = dict(
    lik05=None,
    lik075=None,
    phat=None,
    ll=None,
    mu=None,
    var_mle=None,
    var_unb=None,
    p1=None,
    p3=None,
    p2=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"lik05":[0.0625,0.0001],"lik075":[0.10546875,0.0005],"phat":[0.75,0.001],"ll":[-2.24934,0.003],"mu":[4,0],"var_mle":[2.666667,0.002],"var_unb":[4,0.001],"p1":[0.333333,0.002],"p3":[0.5,0.001],"p2":[0,0.0001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Listelerle yaz; `numpy`/`torch`/`statistics` yasak, `math` serbest.

- `mean(xs)`, `var(xs, ddof=0)` — `ddof=1` yansız varyansı versin.
- `bernoulli_ll(ys, p)` — log-likelihood $\sum[y\ln p+(1-y)\ln(1-p)]$. $p\in\{0,1\}$ ve verinin buna aykırı olduğu durumda `-math.inf` döndür (`if` gerekiyor; `math.log(0)` hata verir).
- `categorical_mle(rolls, K)` — 1..K arası değerler için `K` uzunlukta olasılık listesi.

Kontrol ayrıca 2000 atışlık veride çarpımın 0'a düştüğünü ama log toplamının sonlu kaldığını gösterir.

In [ ]:
%%writefile stage_b.py
import math


def mean(xs):
    ...


def var(xs, ddof=0):
    ...


def bernoulli_ll(ys, p):
    ...


def categorical_mle(rolls, K):
    ...


print(mean([2, 4, 6]), var([2, 4, 6]), var([2, 4, 6], ddof=1))   # 4.0 2.666… 4.0
print(bernoulli_ll([1, 0, 1, 1], 0.75))                          # -2.2493…
print(categorical_mle([1, 3, 3, 6, 3, 1], 6))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da iki iş, ikisi de **döngüsüz**:

- `gaussian_mle(X)` — `X: (n, d)`; her sütun için `(mu: (d,), var: (d,))`, MLE varyansı (`ddof=0`).
- `grid_search_p(ys, grid)` — `grid: (m,)` olası $p$ değerleri. Log-likelihood'u **tüm ızgara için tek ifadede** hesapla: `ys` `(n,)`, `grid[:, None]` `(m, 1)` → broadcasting ile `(m, n)` matris, satır toplamı `(m,)`. En iyi $p$'yi döndür. Izgarada 0 ve 1 olmayacak.
- `class_stats(df, by, col)` — pandas: `by` sütununa göre gruplayıp `col` için `count`, `mean`, `var` (MLE, `ddof=0`) sütunlarından oluşan bir `DataFrame`. İpucu: `groupby(...).agg(...)` ya da `.agg(var=(col, lambda s: s.var(ddof=0)))`.

In [ ]:
%%writefile stage_c.py
import numpy as np
import pandas as pd


def gaussian_mle(X):
    """X: (n, d) → (mu: (d,), var: (d,)). Döngü yok."""
    ...


def grid_search_p(ys, grid):
    """En yüksek Bernoulli log-likelihood'u veren p. Döngü yok."""
    ...


def class_stats(df, by, col):
    """by'a göre grupla; col için count, mean, var (ddof=0) sütunları."""
    ...


print(gaussian_mle(np.array([[2.0, 1.0], [4.0, 1.0], [6.0, 4.0]])))
print(grid_search_p(np.array([1, 0, 1, 1]), np.linspace(0.01, 0.99, 99)))   # 0.75
df = pd.DataFrame({"sinif": ["a", "a", "b", "b", "b"], "boy": [1.0, 3.0, 2.0, 4.0, 6.0]})
print(class_stats(df, "sinif", "boy"))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`torch.distributions` her dağılım için `log_prob` verir; `log_prob` autograd ile türevlenebilir. Böylece kapalı formu olmayan modellerde de MLE'yi gradyanla yapabiliriz. Burada kapalı form var; amacımız iki yolun aynı yere vardığını görmek.

$\sigma>0$ kısıtını gradient descent'e zorla uydurmak yerine **yeniden parametrelendir**: $\sigma=e^{s}$, $s\in\mathbb R$ serbest.

- `normal_nll(x, mu, sigma)` — `torch.distributions.Normal` ile **ortalama** NLL.
- `fit_normal(x, steps=2000, lr=0.05)` → `(mu, sigma)`; `mu` ve `s` parametrelerini `torch.optim.Adam` ile eğit. Kontrol sonucu $\hat\mu=\bar x$ ve $\hat\sigma=\sqrt{\hat\sigma^2_{\text{MLE}}}$ ile karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch
from torch.distributions import Normal


def normal_nll(x, mu, sigma):
    """Ortalama negatif log-likelihood."""
    ...


def fit_normal(x, steps=2000, lr=0.05):
    """σ = exp(s) ile Adam → (mu, sigma) skaler tensörler."""
    mu = torch.zeros((), requires_grad=True)
    s = torch.zeros((), requires_grad=True)
    ...


x = torch.tensor([2.0, 4.0, 6.0])
print(normal_nll(x, torch.tensor(4.0), torch.tensor(1.0)))
print(fit_normal(x))     # ≈ (4.0, 1.633)

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Bootstrap**: veriden yerine koyarak $n$ örnek çek, istatistiği hesapla, bunu $B$ kez tekrarla. $B$ istatistiğin dağılımı, tahminin belirsizliğini verir: ortalamanın standart hatası teoride $\sigma/\sqrt n$.

Döngüyle $B$ kez çekmek yerine tüm indisleri bir kerede çek: `idx = torch.randint(n, (B, n))`, sonra `x[idx]` `(B, n)` ve satır ortalaması.

- `bootstrap_means(x, B, generator)` → `(B,)`. **Döngü yok.** Rastgelelik için verilen `generator`'ı kullan.
- `batched_gaussian_mle(X)` — `X: (B, n)`, her satır ayrı bir veri seti; `(mu: (B,), var: (B,))`. **Döngü yok.**

Kontrol, bootstrap standart sapmasını teorik $\hat\sigma/\sqrt n$ ile karşılaştırır ve döngülü sürümle hız tablosu üretir.

In [ ]:
%%writefile stage_e.py
import torch


def bootstrap_means(x, B, generator):
    """→ (B,) bootstrap ortalamaları. Döngü yok."""
    ...


def batched_gaussian_mle(X):
    """X: (B, n) → (mu: (B,), var: (B,)). Döngü yok."""
    ...


g = torch.Generator().manual_seed(0)
x = torch.randn(100, generator=g) * 2 + 5
m = bootstrap_means(x, 10000, g)
print(m.mean(), m.std(), x.std(unbiased=False) / 10)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Box & Muller (1958): normal sapmaların üretimi

[Box & Muller — A Note on the Generation of Random Normal Deviates (Annals of Mathematical Statistics 1958)](https://projecteuclid.org/journals/annals-of-mathematical-statistics/volume-29/issue-2/A-Note-on-the-Generation-of-Random-Normal-Deviates/10.1214/aoms/1177706645.full)

Box–Muller dönüşümünü ve Marsaglia'nın kutupsal yöntemini yalnız düzgün sayılardan üret; Kolmogorov–Smirnov ve moment testleriyle N(0, 1)'e uyumu, kuyruk olasılıklarını (|z| > 4) ve hız farkını (trigonometri vs ret örneklemesi) ölç. Ters CDF yöntemiyle (Beasley–Springer–Moro ya da kendi rasyonel yaklaşımın) karşılaştır.


**Adımlar**

1. Kutupsal yöntemin kabul oranının π/4 olduğunu doğrula.

**Kabul ölçütleri**

- Üç yöntem de KS testinden geçer.
- Hız ve kuyruk doğruluğu tablosu.

## Kaynaklar

- [Mathematics for Machine Learning, Bölüm 6 (Probability) ve 8.3 (Parameter Estimation)](https://mml-book.github.io/) — © (PDF ücretsiz)
- [Murphy — Probabilistic Machine Learning: An Introduction, Bölüm 2 ve 4](https://probml.github.io/pml-book/book1.html) — CC BY-NC-ND
- [Dive into Deep Learning §2.6 Probability and Statistics](https://d2l.ai/chapter_preliminaries/probability.html) — CC BY-SA 4.0 / MIT-0
- [PyTorch — torch.distributions](https://docs.pytorch.org/docs/stable/distributions.html) — BSD-3